<a href="https://colab.research.google.com/github/mro9395/risk_zones/blob/main/risk_polygon_population_cluster_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Citywide high-risk polygons: population, clusters, and a site draw

This notebook screens **all CDMX high-risk polygons** in `susceptibilidad_laderas_all.geojson` against the 2026 WorldPop population-count raster. It first splits multipart features into individual Polygon parts, retains parts above editable population and density thresholds, clusters nearby retained parts, and selects five primary clusters with no reserves by default. The population threshold defaults to **strictly above 30 estimated people** per part; density is estimated people divided by polygon area in km² and defaults to **strictly above zero people/km²**. It does **not** sample cadastre parcels or count households.

WorldPop's raster values are estimated **people per pixel**, not population density. The zonal sum multiplies each pixel count by the fraction of its area covered by a polygon. These are model-based estimates, especially uncertain for small polygons. A positive value is a screening indicator, not proof that a household occupies the polygon. [WorldPop product metadata](https://stac.worldpop.org/collections/MEX/items/mex_pop_2026_CN_100m_R2025A_v1) · [exactextract sum definition](https://isciences.github.io/exactextract/operations.html).

The source GeoJSON is treated as already limited to high-risk polygons. No additional risk-category filter or four-alcaldía clipping is applied. The selected clusters are **candidate study sites** for later physical inspection, not an automatically approved fieldwork sample. If the corrected 5 m frame contains exactly five clusters, all five are selected and the seed changes review order only.


In [ ]:
%pip -q install geopandas pyogrio shapely pyproj rasterio exactextract matplotlib folium

from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import CRS
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window, from_bounds, bounds as window_bounds
from shapely.geometry import box
from shapely.ops import unary_union
from exactextract import exact_extract
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import folium

try:
    from google.colab import drive
except ImportError:
    drive = None

# Set input names and the site-draw design here.
RAW = Path('/content/drive/MyDrive/Dissertation/data/raw')
WORK = Path('/content/drive/MyDrive/Dissertation/data/output')
BOUNDARY_FILE_NAME = 'susceptibilidad_laderas_all.geojson'
POPULATION_RASTER_STEM = 'mex_pop_2026_CN_100m_R2025A_v1'
MIN_POLYGON_POPULATION = 30.0  # retain estimates strictly above this value
MIN_POLYGON_DENSITY_PEOPLE_KM2 = 0.0  # retain density strictly above this value
CLUSTER_GAPS_M = (5, 25, 50, 100, 200)
CLUSTER_DISTANCE_METHOD = 'perimeter'  # 'perimeter' or 'centroid'
DRAW_CLUSTER_GAP_M = 5
N_PRIMARY_CLUSTERS = 5
N_RESERVE_CLUSTERS = 0
RANDOM_SEED = 20261109
METRIC_CRS = 'EPSG:32614'
metric_crs = CRS.from_user_input(METRIC_CRS)
if not metric_crs.is_projected or not np.isclose(metric_crs.axis_info[0].unit_conversion_factor, 1):
    raise ValueError('Clustering CRS must be projected and measured in metres.')

if drive is not None and not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=False)
if not RAW.exists() or not WORK.exists():
    raise FileNotFoundError(f'Expected existing Drive folders: {RAW} and {WORK}')
if not np.isfinite(MIN_POLYGON_POPULATION) or MIN_POLYGON_POPULATION < 0:
    raise ValueError('MIN_POLYGON_POPULATION must be finite and nonnegative.')
if not np.isfinite(MIN_POLYGON_DENSITY_PEOPLE_KM2) or MIN_POLYGON_DENSITY_PEOPLE_KM2 < 0:
    raise ValueError('MIN_POLYGON_DENSITY_PEOPLE_KM2 must be finite and nonnegative.')
if CLUSTER_DISTANCE_METHOD not in {'perimeter', 'centroid'}:
    raise ValueError('CLUSTER_DISTANCE_METHOD must be perimeter or centroid.')
if DRAW_CLUSTER_GAP_M not in CLUSTER_GAPS_M or any(gap <= 0 for gap in CLUSTER_GAPS_M):
    raise ValueError('The draw gap must be one of the positive sensitivity gaps.')
if N_PRIMARY_CLUSTERS < 1 or N_RESERVE_CLUSTERS < 0:
    raise ValueError('Request at least one primary cluster and no negative reserve count.')

def unique_input(filename, suffixes):
    '''Find exactly one named file under RAW; fail on missing or duplicate copies.'''
    matches = sorted(path for path in RAW.rglob('*') if path.is_file()
                     and path.suffix.lower() in suffixes
                     and (path.name.lower() == filename.lower()
                          or path.stem.lower() == filename.lower()))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected one {filename} under {RAW}; found {matches}')
    return matches[0]

BOUNDARY_PATH = unique_input(BOUNDARY_FILE_NAME, {'.geojson'})
RASTER_PATH = unique_input(POPULATION_RASTER_STEM, {'.tif', '.tiff'})
print('High-risk polygons:', BOUNDARY_PATH)
print('Population raster:', RASTER_PATH)


## 1. Read the citywide boundaries and intersect the raster

Each source feature must be a polygon or multipolygon. Multipart features are split into individual Polygon parts **before** population and density screening and clustering. Each part receives a new `risk_polygon_id`; the original `source_risk_polygon_id`, source row, and part number remain in the outputs. Keep the same source file and feature order to reproduce part IDs. Density uses that part's metric area in km². Missing raster coverage is reported separately from a valid zero population estimate. The first reported total is estimated population across the dissolved union of **all** input boundaries, before either screen.


In [ ]:
source = gpd.read_file(BOUNDARY_PATH)
if source.crs is None:
    raise ValueError(f'{BOUNDARY_PATH.name} has no CRS.')
if source.empty or source.geometry.isna().any() or source.geometry.is_empty.any():
    raise ValueError('The citywide boundary layer is empty or contains missing geometries.')
source_bounds_wgs84 = source.to_crs(4326).total_bounds
if not (-101 < source_bounds_wgs84[0] < source_bounds_wgs84[2] < -97
        and 18 < source_bounds_wgs84[1] < source_bounds_wgs84[3] < 21):
    raise ValueError(f'Source bounds {source_bounds_wgs84} do not look like CDMX; check its CRS.')
print(f'Source CRS: {source.crs}; clustering CRS: {METRIC_CRS} (metres)')
source = source.reset_index(drop=True).copy()
source['source_feature_index'] = np.arange(1, len(source) + 1)
source['geometry'] = source.geometry.make_valid()
if not source.geometry.geom_type.isin(['Polygon', 'MultiPolygon']).all():
    kinds = source.geometry.geom_type.value_counts().to_dict()
    raise ValueError(f'Expected polygonal high-risk features only; found {kinds}')
source_feature_count = len(source)
multipart_feature_count = int(source.geometry.geom_type.eq('MultiPolygon').sum())
expected_part_count = sum(len(geom.geoms) if geom.geom_type == 'MultiPolygon' else 1
                          for geom in source.geometry)
if 'risk_polygon_id' in source.columns:
    ids = source.risk_polygon_id.astype(str).str.strip()
    if ids.eq('').any() or source.risk_polygon_id.isna().any() or ids.duplicated().any():
        raise ValueError('risk_polygon_id must be complete and unique in the source file.')
else:
    ids = pd.Series([f'S{i:06d}' for i in source.source_feature_index])

source['source_risk_polygon_id'] = ids.to_numpy()
# One multipart feature can contain pieces kilometres apart. Treat each piece
# as a separate polygon before measuring population or geographic distance.
parts = source[['source_feature_index', 'source_risk_polygon_id', 'geometry']].explode(
    index_parts=False, ignore_index=True
)
if len(parts) != expected_part_count or not parts.geometry.geom_type.eq('Polygon').all():
    raise ValueError('Exploding the source did not yield one Polygon per part.')
parts['source_part_number'] = parts.groupby('source_feature_index', sort=False).cumcount() + 1
parts['risk_polygon_id'] = [f'P{i:07d}' for i in range(1, len(parts) + 1)]
polygons = parts[[
    'risk_polygon_id', 'source_risk_polygon_id', 'source_feature_index',
    'source_part_number', 'geometry',
]].to_crs(METRIC_CRS)
print(f'Source features: {source_feature_count:,}; multipart features: {multipart_feature_count:,}; '
      f'individual polygon parts: {len(polygons):,}')
polygons['polygon_area_m2'] = polygons.geometry.area
if polygons.polygon_area_m2.le(0).any():
    raise ValueError('Risk polygons include zero-area geometries after repair.')

with rasterio.open(RASTER_PATH) as raster:
    if raster.count != 1 or raster.crs is None or raster.nodata is None:
        raise ValueError('Expected a single-band georeferenced population raster with nodata.')
    raster_crs = raster.crs
    if not box(*raster.bounds).intersects(box(*polygons.to_crs(raster_crs).total_bounds)):
        raise ValueError('Population raster does not overlap CDMX polygons; check both CRSs.')
    print(f'Raster CRS: {raster_crs}; cell size: {raster.res}; nodata: {raster.nodata}')
    # exactextract performs an area-fraction-weighted sum of people-per-cell values.
    polygon_stats = exact_extract(
        raster, polygons.to_crs(raster_crs),
        ['population_estimate=sum', 'valid_cell_coverage=count'],
        include_cols='risk_polygon_id', output='pandas',
    )

if polygon_stats.risk_polygon_id.duplicated().any() or len(polygon_stats) != len(polygons):
    raise AssertionError('Raster extraction did not return one row per source polygon.')
polygons = polygons.merge(polygon_stats, on='risk_polygon_id', validate='one_to_one')
polygons['valid_cell_coverage'] = polygons.valid_cell_coverage.fillna(0)
polygons['population_estimate'] = pd.to_numeric(polygons.population_estimate, errors='coerce')
if polygons.population_estimate.dropna().lt(-1e-6).any():
    raise ValueError('The population raster has negative values beyond nodata.')
polygons['population_density_people_km2'] = (
    polygons.population_estimate / (polygons.polygon_area_m2 / 1_000_000)
)

def population_on_union(frame):
    '''Sum fractional raster-cell population on the dissolved polygon union once.'''
    if frame.empty:
        return 0.0
    dissolved = gpd.GeoDataFrame(geometry=[unary_union(frame.geometry.tolist())], crs=frame.crs)
    with rasterio.open(RASTER_PATH) as population_raster:
        result = exact_extract(population_raster, dissolved.to_crs(population_raster.crs),
                               'sum', output='pandas')
    estimate = float(result['sum'].iloc[0])
    if not np.isfinite(estimate):
        raise ValueError('The dissolved boundaries have no valid population raster coverage.')
    return estimate

all_boundary_population_estimate = population_on_union(polygons)
print(f'All input boundaries before filtering: {all_boundary_population_estimate:,.1f} estimated people '
      '(overlapping polygons counted once)')
polygons['screening_status'] = np.select(
    [polygons.valid_cell_coverage.le(0) | polygons.population_estimate.isna(),
     polygons.population_estimate.le(MIN_POLYGON_POPULATION),
     polygons.population_density_people_km2.le(MIN_POLYGON_DENSITY_PEOPLE_KM2)],
    ['no_raster_data', 'below_or_equal_population_threshold',
     'below_or_equal_density_threshold'], default='eligible',
)
eligible_polygons = polygons[polygons.screening_status.eq('eligible')].copy().reset_index(drop=True)
if eligible_polygons.empty:
    raise ValueError('No polygons passed the population and density screens. Inspect raster coverage and thresholds.')
print(f'Source features: {source_feature_count:,}; individual polygon parts: {len(polygons):,}')
print('Population and density screening status:', polygons.screening_status.value_counts().to_dict())
display(polygons.drop(columns='geometry').head(10))

polygons.to_crs(4326).to_file(WORK / 'citywide_risk_polygon_population_screen.geojson', driver='GeoJSON')
polygons.drop(columns='geometry').to_csv(WORK / 'citywide_risk_polygon_population_screen.csv', index=False)
eligible_polygons.to_crs(4326).to_file(WORK / 'citywide_population_eligible_polygons.geojson', driver='GeoJSON')


## 2. Cluster retained polygons

Single-part polygons connect when their exact edge-to-edge distance is at most the specified gap in EPSG:32614 metres. Connected chains form one cluster; member polygons are dissolved before estimating **cluster population** so overlapping risk polygons cannot double count raster cells. Cluster density is this population divided by the **dissolved cluster area in km²**. The output reports area, density, and each cluster's bounding-box width, height, and diagonal in metres. A chain can span over 1 km even when every individual link is within 5 m; no maximum span rule is imposed. The five gap scenarios are screening alternatives. Only `DRAW_CLUSTER_GAP_M` feeds the draw. The optional centroid metric remains available for sensitivity.


In [ ]:
def make_clusters(frame, gap_m, method):
    '''Build connected components from perimeter or centroid distance.'''
    work = frame.sort_values('risk_polygon_id').reset_index(drop=True).copy()
    comparison = work.geometry.centroid if method == 'centroid' else work.geometry
    if comparison.crs != metric_crs:
        raise ValueError(f'Clustering requires {METRIC_CRS}; got {comparison.crs}.')
    parent = list(range(len(work)))

    def find(item):
        '''Find a component root while compressing the union-find path.'''
        while parent[item] != item:
            parent[item] = parent[parent[item]]
            item = parent[item]
        return item

    if 'dwithin' in comparison.sindex.valid_query_predicates:
        # This checks exact geometry distance in metres, rather than buffered
        # envelope overlap or distance between source MultiPolygon features.
        pairs = comparison.sindex.query(comparison, predicate='dwithin', distance=gap_m)
    else:
        # Fallback for older spatial indexes: buffer only to find candidates,
        # then verify every candidate with exact geometry distance below.
        expanded = gpd.GeoSeries(comparison.buffer(gap_m / 2), crs=work.crs)
        pairs = expanded.sindex.query(expanded, predicate='intersects')
    for left, right in zip(*pairs):
        left, right = int(left), int(right)
        if left < right and comparison.iloc[left].distance(comparison.iloc[right]) <= gap_m:
            root_left, root_right = find(left), find(right)
            if root_left != root_right:
                parent[root_right] = root_left

    work['component_root'] = [find(i) for i in range(len(work))]
    records, members = [], []
    # Sorting by the smallest member ID stabilizes labels for the same source file.
    groups = sorted((group for _, group in work.groupby('component_root')),
                    key=lambda group: group.risk_polygon_id.min())
    for number, group in enumerate(groups, start=1):
        cluster_id = f'{method[:3].upper()}_{gap_m:03d}m_C{number:05d}'
        ids = sorted(group.risk_polygon_id.tolist())
        records.append({
            'cluster_id': cluster_id,
            'cluster_gap_m': gap_m,
            'cluster_distance_method': method,
            'risk_polygon_count': len(ids),
            'source_feature_count': group.source_feature_index.nunique(),
            'member_polygon_ids': ','.join(ids),
            'geometry': unary_union(group.geometry.tolist()),
        })
        members.extend({'cluster_id': cluster_id, 'risk_polygon_id': polygon_id,
                        'cluster_gap_m': gap_m} for polygon_id in ids)
    return gpd.GeoDataFrame(records, geometry='geometry', crs=frame.crs), pd.DataFrame(members)

# Synthetic check: distant pieces from one MultiPolygon must stay separate,
# while two single polygons with a 4 m edge gap must connect at 5 m.
test_source = gpd.GeoDataFrame({
    'source_feature_index': [1],
    'geometry': [unary_union([box(0, 0, 10, 10), box(2000, 0, 2010, 10)])],
}, geometry='geometry', crs=METRIC_CRS)
test_parts = test_source.explode(index_parts=False, ignore_index=True)
test_parts['risk_polygon_id'] = ['test_far_1', 'test_far_2']
if len(make_clusters(test_parts, 5, 'perimeter')[0]) != 2:
    raise AssertionError('Distant MultiPolygon parts incorrectly joined at 5 m.')
test_near = gpd.GeoDataFrame({
    'source_feature_index': [1, 2],
    'risk_polygon_id': ['test_near_1', 'test_near_2'],
    'geometry': [box(0, 0, 10, 10), box(14, 0, 24, 10)],
}, geometry='geometry', crs=METRIC_CRS)
if len(make_clusters(test_near, 5, 'perimeter')[0]) != 1:
    raise AssertionError('Polygons 4 m apart failed to connect at 5 m.')
print('Distance checks passed: 4 m connects; distant multipart pieces stay separate.')

cluster_parts, membership_parts = [], []
for gap in CLUSTER_GAPS_M:
    one_scenario, one_membership = make_clusters(eligible_polygons, gap, CLUSTER_DISTANCE_METHOD)
    with rasterio.open(RASTER_PATH) as raster:
        cluster_stats = exact_extract(
            raster, one_scenario.to_crs(raster.crs),
            ['cluster_population_estimate=sum', 'valid_cell_coverage=count'],
            include_cols='cluster_id', output='pandas',
        )
    one_scenario = one_scenario.merge(cluster_stats, on='cluster_id', validate='one_to_one')
    if one_scenario.cluster_population_estimate.isna().any():
        raise ValueError(f'Cluster population extraction has missing values at {gap} m.')
    one_scenario['cluster_area_m2'] = one_scenario.geometry.area
    if one_scenario.cluster_area_m2.le(0).any():
        raise ValueError(f'Clusters include zero-area geometry at {gap} m.')
    one_scenario['cluster_area_km2'] = one_scenario.cluster_area_m2 / 1_000_000
    one_scenario['cluster_density_people_km2'] = (
        one_scenario.cluster_population_estimate / one_scenario.cluster_area_km2
    )
    cluster_bounds = one_scenario.geometry.bounds
    one_scenario['cluster_bbox_width_m'] = cluster_bounds.maxx - cluster_bounds.minx
    one_scenario['cluster_bbox_height_m'] = cluster_bounds.maxy - cluster_bounds.miny
    one_scenario['cluster_bbox_diagonal_m'] = np.hypot(
        one_scenario.cluster_bbox_width_m, one_scenario.cluster_bbox_height_m
    )
    cluster_parts.append(one_scenario)
    membership_parts.append(one_membership)
    print(f'{gap} m {CLUSTER_DISTANCE_METHOD}: {len(one_scenario):,} clusters')

clusters = gpd.GeoDataFrame(pd.concat(cluster_parts, ignore_index=True),
                            geometry='geometry', crs=METRIC_CRS)
membership = pd.concat(membership_parts, ignore_index=True)
if membership.duplicated(['cluster_gap_m', 'risk_polygon_id']).any():
    raise AssertionError('A polygon belongs to more than one cluster in one gap scenario.')
expected_ids = set(eligible_polygons.risk_polygon_id)
for gap in CLUSTER_GAPS_M:
    observed_ids = set(membership.loc[membership.cluster_gap_m.eq(gap), 'risk_polygon_id'])
    if observed_ids != expected_ids:
        raise AssertionError(f'Population-eligible polygons are missing or duplicated at {gap} m.')
clusters.to_crs(4326).to_file(WORK / 'citywide_population_risk_clusters.geojson', driver='GeoJSON')
clusters.drop(columns='geometry').to_csv(WORK / 'citywide_population_risk_clusters.csv', index=False)
membership.to_csv(WORK / 'citywide_population_cluster_membership.csv', index=False)
display(clusters.drop(columns='geometry').groupby('cluster_gap_m').agg(
    clusters=('cluster_id', 'size'), polygons=('risk_polygon_count', 'sum'),
    median_population=('cluster_population_estimate', 'median'),
    median_area_km2=('cluster_area_km2', 'median'),
    median_density_people_km2=('cluster_density_people_km2', 'median')).round({
        'median_population': 1, 'median_area_km2': 6,
        'median_density_people_km2': 1,
    }))


## 3. Interactive cluster map

This map shows every dissolved cluster at the selected gap, colour coded for navigation. Hover or click a cluster to see its ID, single-part polygon count, source-feature count, estimated population, dissolved boundary area in km², estimated people per km², and bounding-box span. Density divides the cluster's independently extracted population by its **dissolved high-risk area**, not by its bounding box. The layer control switches between OpenTopoMap (streets and terrain) and Esri World Imagery, and shows or hides all clusters. Up to 20 of the most populous clusters have visible labels to keep large maps readable. It runs directly after clustering and saves an HTML map to Drive. The basemap tiles require an internet connection when viewing the saved HTML; if a provider is blocked in your browser, try the other basemap in the layer control.


In [ ]:
# Build the interactive map from the dissolved cluster geometries in WGS 84.
map_clusters = clusters[clusters.cluster_gap_m.eq(DRAW_CLUSTER_GAP_M)].to_crs(4326)
if map_clusters.empty:
    raise ValueError(f'No clusters exist at the {DRAW_CLUSTER_GAP_M} m map gap.')
map_bounds = map_clusters.total_bounds  # west, south, east, north
cluster_map = folium.Map(
    location=[(map_bounds[1] + map_bounds[3]) / 2,
              (map_bounds[0] + map_bounds[2]) / 2],
    zoom_start=10, tiles=None, control_scale=True,
)
# Offer two distinct basemap hosts; switch layers if one is blocked.
# Both services require live internet when viewing the HTML map.
folium.TileLayer(
    tiles='https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png',
    attr='Map data: &copy; OpenStreetMap contributors, SRTM | Map style: &copy; OpenTopoMap (CC-BY-SA)',
    name='OpenTopoMap: streets and terrain', subdomains='abc',
    max_native_zoom=17, max_zoom=19,
    show=True, control=True,
).add_to(cluster_map)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri, Vantor, Earthstar Geographics, and the GIS User Community',
    name='Esri World Imagery: satellite', max_zoom=19,
    show=False, control=True,
).add_to(cluster_map)
# Put all clusters in one GeoJSON layer so the HTML remains usable when
# splitting multipart source features produces many more than five clusters.
map_features = map_clusters[[
    'cluster_id', 'risk_polygon_count', 'source_feature_count',
    'cluster_population_estimate', 'cluster_area_km2',
    'cluster_density_people_km2', 'cluster_bbox_diagonal_m', 'geometry',
]].copy().rename(columns={
    'cluster_population_estimate': 'population_estimate',
    'cluster_area_km2': 'area_km2',
    'cluster_density_people_km2': 'density_people_km2',
    'cluster_bbox_diagonal_m': 'bbox_diagonal_m',
})
map_features['population_estimate'] = map_features.population_estimate.round(1)
map_features['area_km2'] = map_features.area_km2.round(6)
map_features['density_people_km2'] = map_features.density_people_km2.round(1)
map_features['bbox_diagonal_m'] = map_features.bbox_diagonal_m.round(1)
colours_by_id = {
    cluster_id: to_hex(plt.get_cmap('tab20')(index % 20 / 20))
    for index, cluster_id in enumerate(sorted(map_features.cluster_id))
}
folium.GeoJson(
    map_features.to_json(),
    name=f'All {len(map_features):,} clusters at {DRAW_CLUSTER_GAP_M} m',
    style_function=lambda feature: {
        'color': colours_by_id[feature['properties']['cluster_id']],
        'weight': 1.5,
        'fillColor': colours_by_id[feature['properties']['cluster_id']],
        'fillOpacity': 0.35,
    },
    highlight_function=lambda _: {'weight': 3, 'fillOpacity': 0.55},
    tooltip=folium.GeoJsonTooltip(
        fields=['cluster_id', 'risk_polygon_count', 'source_feature_count',
                'population_estimate', 'area_km2', 'density_people_km2',
                'bbox_diagonal_m'],
        aliases=['Cluster', 'Polygon parts', 'Source features',
                 'Estimated people', 'Dissolved area (km²)',
                 'Estimated people/km²', 'Bounding-box diagonal (m)'],
    ),
    popup=folium.GeoJsonPopup(
        fields=['cluster_id', 'risk_polygon_count', 'source_feature_count',
                'population_estimate', 'area_km2', 'density_people_km2',
                'bbox_diagonal_m'],
        aliases=['Cluster', 'Polygon parts', 'Source features',
                 'Estimated people', 'Dissolved area (km²)',
                 'Estimated people/km²', 'Bounding-box diagonal (m)'],
    ),
    zoom_on_click=True,
).add_to(cluster_map)
# Label only the largest population clusters so labels do not cover the map.
for row in map_features.nlargest(min(20, len(map_features)),
                                 'population_estimate').itertuples(index=False):
    anchor = row.geometry.representative_point()
    short_id = row.cluster_id.rsplit('_', 1)[-1]
    folium.Marker(
        [anchor.y, anchor.x],
        icon=folium.DivIcon(html=(
            '<div style="font-size:10px;font-weight:bold;color:#111;'
            'background:white;padding:1px;border:1px solid #555;">'
            f'{short_id}</div>'
        )),
    ).add_to(cluster_map)
cluster_map.fit_bounds([
    [map_bounds[1], map_bounds[0]],
    [map_bounds[3], map_bounds[2]],
])
folium.LayerControl(collapsed=False).add_to(cluster_map)
map_path = WORK / f'citywide_cluster_map_{DRAW_CLUSTER_GAP_M}m.html'
cluster_map.save(str(map_path))
print(f'Interactive cluster map: {map_path}')
display(cluster_map)


## 4. Inspect clusters before selection

These plots run before the draw. They show source-feature and polygon-part counts, member parts and population by cluster, cluster spans, a raster-backed overview with density labels for up to 20 populous clusters, and maps for every gap scenario with median cluster density in each title. PNG copies are saved in the output folder. The full cluster table and interactive map contain every cluster.


In [ ]:
def raster_preview(ax, metric_bounds, max_side=950):
    '''Read only a bounded, downsampled raster window for a map preview.'''
    metric_box = gpd.GeoSeries([box(*metric_bounds)], crs=METRIC_CRS)
    left, bottom, right, top = metric_box.to_crs(raster_crs).total_bounds
    with rasterio.open(RASTER_PATH) as raster:
        window = from_bounds(left, bottom, right, top, raster.transform)
        window = window.intersection(Window(0, 0, raster.width, raster.height))
        width, height = max(1, round(window.width)), max(1, round(window.height))
        scale = min(1, max_side / max(width, height))
        out_shape = (max(1, round(height * scale)), max(1, round(width * scale)))
        image = raster.read(1, window=window, out_shape=out_shape,
                            resampling=Resampling.average, masked=True)
        extent = window_bounds(window, raster.transform)
    # Log colour scaling keeps both sparse and dense cells visible.
    image = np.log1p(np.ma.maximum(image, 0))
    return ax.imshow(image, extent=extent, origin='upper', cmap='viridis', alpha=0.75)

# These figures run before the site draw, including when requested counts are infeasible.
scenario = clusters[clusters.cluster_gap_m.eq(DRAW_CLUSTER_GAP_M)].copy()
scenario_wgs84 = scenario.to_crs(4326)
source_wgs84 = polygons.to_crs(4326)
print(f'Source features: {source_feature_count:,}; multipart: {multipart_feature_count:,}')
print(f'Individual polygon parts: {len(polygons):,}; population-and-density eligible: {len(eligible_polygons):,}')
print(f'Clusters at {DRAW_CLUSTER_GAP_M} m: {len(scenario):,}')
print(f'Clusters with bounding-box diagonal over 1 km: '
      f'{int(scenario.cluster_bbox_diagonal_m.gt(1000).sum()):,}')
display(scenario.drop(columns='geometry')[[
    'cluster_id', 'risk_polygon_count', 'source_feature_count',
    'cluster_population_estimate', 'cluster_area_km2',
    'cluster_density_people_km2', 'cluster_bbox_width_m',
    'cluster_bbox_height_m', 'cluster_bbox_diagonal_m',
]].sort_values('cluster_bbox_diagonal_m', ascending=False).head(30))

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
gap_counts = clusters.groupby('cluster_gap_m').size().reindex(CLUSTER_GAPS_M)
axes[0].bar([str(gap) for gap in CLUSTER_GAPS_M], gap_counts.values, color='#3182bd')
axes[0].set(xlabel='Connection gap (m)', ylabel='Number of clusters',
            title='How the gap changes the cluster count')
ordered = scenario.nlargest(20, 'risk_polygon_count').sort_values('risk_polygon_count')
axes[1].barh(ordered.cluster_id, ordered.risk_polygon_count, color='#756bb1')
axes[1].set(xlabel='Polygon parts', title='Largest 20 clusters by parts')
ordered_pop = scenario.nlargest(20, 'cluster_population_estimate').sort_values(
    'cluster_population_estimate')
axes[2].barh(ordered_pop.cluster_id, ordered_pop.cluster_population_estimate,
             color='#31a354')
axes[2].set(xlabel='Estimated people', title='Largest 20 by population')
ordered_span = scenario.nlargest(20, 'cluster_bbox_diagonal_m').sort_values(
    'cluster_bbox_diagonal_m')
axes[3].barh(ordered_span.cluster_id, ordered_span.cluster_bbox_diagonal_m / 1000,
             color='#e6550d')
axes[3].set(xlabel='Bounding-box diagonal (km)', title='Largest 20 by span')
plt.tight_layout()
fig.savefig(WORK / 'citywide_cluster_size_diagnostics.png', dpi=180, bbox_inches='tight')
plt.show()

metric_bounds = polygons.total_bounds
pad_m = 1000
metric_bounds = (metric_bounds[0]-pad_m, metric_bounds[1]-pad_m,
                 metric_bounds[2]+pad_m, metric_bounds[3]+pad_m)
fig, ax = plt.subplots(figsize=(11, 10))
image = raster_preview(ax, metric_bounds)
source_wgs84.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.2, alpha=0.5)
scenario_wgs84.boundary.plot(ax=ax, color='#de2d26', linewidth=0.7)
for row in scenario_wgs84.nlargest(
        min(20, len(scenario_wgs84)), 'cluster_population_estimate').itertuples(index=False):
    anchor = row.geometry.representative_point()
    ax.text(anchor.x, anchor.y,
            f"{row.cluster_id.rsplit('_', 1)[-1]}\n{row.cluster_density_people_km2:,.0f} people/km²",
            fontsize=7, weight='bold',
            bbox={'facecolor': 'white', 'alpha': 0.8, 'pad': 1})
ax.legend(handles=[Line2D([0], [0], color='#de2d26', lw=1.5,
                          label=f'All {len(scenario_wgs84):,} cluster boundaries')],
          loc='upper left')
fig.colorbar(image, ax=ax, shrink=0.7,
             label='log(1 + average estimated people per source cell)')
ax.set(title=f'High-risk cluster boundaries at {DRAW_CLUSTER_GAP_M} m',
       xlabel='Longitude', ylabel='Latitude')
plt.tight_layout()
fig.savefig(WORK / 'citywide_cluster_map_before_draw.png', dpi=180, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(17, 10), squeeze=False)
display_bounds = source_wgs84.total_bounds
for ax, gap in zip(axes.flat, CLUSTER_GAPS_M):
    source_wgs84.boundary.plot(ax=ax, color='#d0d0d0', linewidth=0.25)
    one_gap = clusters[clusters.cluster_gap_m.eq(gap)].to_crs(4326)
    one_gap.boundary.plot(ax=ax, color='#1f78b4', linewidth=0.6)
    ax.set(title=(f'{gap} m: {len(one_gap)} clusters\n'
                  f'median {one_gap.cluster_density_people_km2.median():,.0f} people/km²'),
           xlabel='Longitude', ylabel='Latitude')
    ax.set_xlim(display_bounds[0], display_bounds[2])
    ax.set_ylim(display_bounds[1], display_bounds[3])
for ax in list(axes.flat)[len(CLUSTER_GAPS_M):]:
    ax.axis('off')
plt.tight_layout()
fig.savefig(WORK / 'citywide_cluster_gap_comparison.png', dpi=180, bbox_inches='tight')
plt.show()


## 5. Order and select candidate sites across four broad geographic groups

The geographic groups are west/east and south/north quadrants split at the median **eligible-cluster representative-point** coordinates for the chosen gap. Within each group, clusters are shuffled with the fixed seed, then ordered round-robin across groups. The first five are primary candidates; reserves are set to zero. If the corrected frame contains more than five clusters, this produces a random geographically spread subset. If it contains exactly five, all are selected and only their review order is randomized. If fewer than five exist, the notebook shows the available count and stops. Selection does not establish physical comparability; review terrain, ravines, roads, and site access afterward.


In [ ]:
draw_frame = clusters[clusters.cluster_gap_m.eq(DRAW_CLUSTER_GAP_M)].copy().reset_index(drop=True)
requested = N_PRIMARY_CLUSTERS + N_RESERVE_CLUSTERS
if len(draw_frame) < requested:
    print('Available clusters by gap:')
    display(clusters.groupby('cluster_gap_m').size().rename('cluster_count').to_frame())
    print(f'Clusters at the selected {DRAW_CLUSTER_GAP_M} m gap:')
    display(draw_frame[['cluster_id', 'risk_polygon_count', 'cluster_population_estimate']])
    raise ValueError(f'Only {len(draw_frame)} clusters exist at {DRAW_CLUSTER_GAP_M} m; '
                     f'{requested} primary + reserve sites were requested. '
                     'Reduce the requested counts or choose a gap with enough clusters.')
anchors = draw_frame.geometry.representative_point()
split_x, split_y = float(anchors.x.median()), float(anchors.y.median())
draw_frame['geographic_group'] = (
    np.where(anchors.y >= split_y, 'north', 'south') + '_' +
    np.where(anchors.x >= split_x, 'east', 'west')
)
print('Draw-frame clusters by geographic group:')
display(draw_frame.geographic_group.value_counts().sort_index().rename('clusters').to_frame())

def spatially_stratified_order(frame, seed):
    '''Shuffle clusters within each quadrant, then alternate among quadrants.'''
    rng = np.random.default_rng(seed)
    groups = {}
    for name, group in frame.groupby('geographic_group', sort=True):
        ids = sorted(group.cluster_id.tolist())
        groups[name] = list(rng.permutation(ids))
    group_order = list(rng.permutation(sorted(groups)))
    result = []
    while any(groups.values()):
        for name in group_order:
            if groups[name]:
                result.append(groups[name].pop())
    if len(result) != len(frame) or len(set(result)) != len(result):
        raise AssertionError('Spatial draw omitted or duplicated a cluster.')
    return result

ordered_ids = spatially_stratified_order(draw_frame, RANDOM_SEED)
rank_lookup = {cluster_id: rank for rank, cluster_id in enumerate(ordered_ids, 1)}
draw_frame['random_order'] = draw_frame.cluster_id.map(rank_lookup)
draw_frame['selection_status'] = np.select(
    [draw_frame.random_order.le(N_PRIMARY_CLUSTERS),
     draw_frame.random_order.le(requested)],
    ['primary', 'reserve'], default='not_selected',
)
selected_clusters = draw_frame[~draw_frame.selection_status.eq('not_selected')].copy()
selected_clusters = selected_clusters.sort_values('random_order')
selected_links = membership[
    membership.cluster_gap_m.eq(DRAW_CLUSTER_GAP_M) &
    membership.cluster_id.isin(selected_clusters.cluster_id)
].merge(selected_clusters[['cluster_id', 'geographic_group', 'selection_status',
                           'random_order']], on='cluster_id', validate='many_to_one')
selected_polygons = eligible_polygons.merge(
    selected_links.drop(columns='cluster_gap_m'), on='risk_polygon_id', validate='one_to_one'
)
if len(selected_polygons) != selected_clusters.risk_polygon_count.sum():
    raise AssertionError('Selected polygon membership does not match the selected clusters.')
selected_boundary_population_estimate = population_on_union(selected_polygons)
print(f'Selected boundaries (primary + reserves): {selected_boundary_population_estimate:,.1f} '
      'estimated people (overlapping polygons counted once)')

draw_frame.to_crs(4326).to_file(WORK / 'citywide_population_cluster_draw_frame.geojson', driver='GeoJSON')
draw_frame.drop(columns='geometry').to_csv(WORK / 'citywide_population_cluster_draw_frame.csv', index=False)
selected_clusters.to_crs(4326).to_file(WORK / 'citywide_population_selected_clusters.geojson', driver='GeoJSON')
selected_clusters.drop(columns='geometry').to_csv(WORK / 'citywide_population_selected_clusters.csv', index=False)
selected_polygons.to_crs(4326).to_file(WORK / 'citywide_population_selected_boundary_polygons.geojson', driver='GeoJSON')
manifest = {
    'boundary_file': str(BOUNDARY_PATH),
    'population_raster': str(RASTER_PATH),
    'population_units': 'estimated people per raster cell',
    'population_method': 'sum of cell counts weighted by cell fraction inside each polygon',
    'source_geometry_rule': 'split each MultiPolygon feature into single Polygon parts before screening',
    'cluster_span_rule': 'report bounding-box width, height, and diagonal; no maximum span cutoff',
    'cluster_distance_crs': METRIC_CRS,
    'cluster_link_rule': 'exact pairwise distance at or below gap; connected components may extend farther',
    'minimum_polygon_population_exclusive': MIN_POLYGON_POPULATION,
    'minimum_polygon_density_people_km2_exclusive': MIN_POLYGON_DENSITY_PEOPLE_KM2,
    'population_density_method': 'area-weighted polygon population divided by polygon area in km2',
    'population_total_method': 'area-weighted raster sum on dissolved boundary union',
    'all_input_boundary_population_estimate': all_boundary_population_estimate,
    'selected_boundary_population_estimate': selected_boundary_population_estimate,
    'cluster_gaps_m': list(CLUSTER_GAPS_M),
    'cluster_distance_method': CLUSTER_DISTANCE_METHOD,
    'draw_cluster_gap_m': DRAW_CLUSTER_GAP_M,
    'primary_clusters': N_PRIMARY_CLUSTERS,
    'reserve_clusters': N_RESERVE_CLUSTERS,
    'random_seed': RANDOM_SEED,
    'geographic_groups': 'four quadrants split at median eligible-cluster representative-point x/y',
    'quadrant_split_x_epsg32614': split_x,
    'quadrant_split_y_epsg32614': split_y,
    'source_feature_count': source_feature_count,
    'multipart_source_feature_count': multipart_feature_count,
    'polygon_part_count': len(polygons),
    'eligible_polygon_count': len(eligible_polygons),
    'draw_frame_cluster_count': len(draw_frame),
}
(WORK / 'citywide_population_cluster_manifest.json').write_text(
    json.dumps(manifest, indent=2), encoding='utf-8'
)
display(selected_clusters[['random_order', 'selection_status', 'cluster_id',
                           'geographic_group', 'risk_polygon_count',
                           'source_feature_count', 'cluster_population_estimate',
                           'cluster_area_km2', 'cluster_density_people_km2',
                           'cluster_bbox_diagonal_m']].reset_index(drop=True))


## 6. Preview source data, screening, and selected sites

The overview uses a downsampled raster **for display only**; extraction above uses the original raster. Red outlines are primary clusters; orange outlines are reserves. The close-up maps show the raster, all local risk polygons, the selected cluster boundary, and its dissolved area and estimated people/km² in the title. The plot and CSVs make the sampling frame and random order auditable before downloading GeoJSONs.


In [ ]:
overview_bounds = polygons.total_bounds
pad = 1000
overview_bounds = (overview_bounds[0]-pad, overview_bounds[1]-pad,
                   overview_bounds[2]+pad, overview_bounds[3]+pad)
fig, ax = plt.subplots(figsize=(11, 10))
image = raster_preview(ax, overview_bounds)
polygons.to_crs(raster_crs).boundary.plot(ax=ax, color='#8a8a8a', linewidth=0.15, alpha=0.6)
eligible_polygons.to_crs(raster_crs).boundary.plot(ax=ax, color='#d9f0a3', linewidth=0.25)
selection_legend = []
for status, colour in [('reserve', '#fe9929'), ('primary', '#de2d26')]:
    subset = selected_clusters[selected_clusters.selection_status.eq(status)]
    if not subset.empty:
        subset.to_crs(raster_crs).boundary.plot(ax=ax, color=colour, linewidth=2)
        selection_legend.append(Line2D([0], [0], color=colour, lw=2,
                                       label=f'{status.title()} clusters'))
ax.set_title('CDMX source polygons, population raster, and selected clusters')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
fig.colorbar(image, ax=ax, shrink=0.65,
             label='log(1 + average estimated people per source cell)')
ax.legend(handles=[
    Patch(color='#8a8a8a', label='All high-risk polygon boundaries'),
    Patch(color='#d9f0a3', label='Population-and-density eligible boundaries'),
] + selection_legend, loc='best')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
status_counts = polygons.screening_status.value_counts().reindex(
    ['eligible', 'below_or_equal_population_threshold',
     'below_or_equal_density_threshold', 'no_raster_data'], fill_value=0)
axes[0].barh(status_counts.index, status_counts.values,
             color=['#31a354', '#9e9ac8', '#bdbdbd', '#de2d26'])
axes[0].invert_yaxis(); axes[0].set_xlabel('High-risk polygons')
axes[0].set_title('Population and density screening')
for row in selected_clusters.itertuples(index=False):
    colour = '#de2d26' if row.selection_status == 'primary' else '#fe9929'
    axes[1].barh(f'{row.random_order}. {row.cluster_id}', row.cluster_population_estimate,
                 color=colour)
axes[1].invert_yaxis(); axes[1].set_xlabel('Estimated population inside dissolved cluster')
axes[1].set_title('Selected clusters')
plt.tight_layout(); plt.show()

map_rows = int(np.ceil(len(selected_clusters) / 2))
fig, axes = plt.subplots(map_rows, 2, figsize=(15, 5 * map_rows), squeeze=False)
for ax, row in zip(axes.flat, selected_clusters.itertuples(index=False)):
    minx, miny, maxx, maxy = row.geometry.bounds
    pad = max(250, 0.12 * max(maxx-minx, maxy-miny))
    bounds = (minx-pad, miny-pad, maxx+pad, maxy+pad)
    raster_preview(ax, bounds, max_side=600)
    local_ids = list(polygons.sindex.query(box(*bounds), predicate='intersects'))
    polygons.iloc[local_ids].to_crs(raster_crs).boundary.plot(
        ax=ax, color='#cccccc', linewidth=0.5)
    members = selected_polygons[selected_polygons.cluster_id.eq(row.cluster_id)]
    members.to_crs(raster_crs).boundary.plot(ax=ax, color='#ffffff', linewidth=1.1)
    gpd.GeoSeries([row.geometry], crs=METRIC_CRS).to_crs(raster_crs).boundary.plot(
        ax=ax, color='#de2d26' if row.selection_status == 'primary' else '#fe9929',
        linewidth=2)
    ax.set_title(f'{row.random_order}. {row.cluster_id} ({row.selection_status}) | '
                 f'{row.risk_polygon_count} parts | ~{row.cluster_population_estimate:,.0f} people\n'
                 f'{row.cluster_area_km2:,.6f} km² | '
                 f'{row.cluster_density_people_km2:,.0f} people/km² | '
                 f'bbox ~{row.cluster_bbox_diagonal_m / 1000:,.1f} km')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
for ax in list(axes.flat)[len(selected_clusters):]:
    ax.axis('off')
plt.tight_layout(); plt.show()


## Outputs and interpretation

- `citywide_risk_polygon_population_screen.csv/.geojson`: every single-part polygon, estimated population, people/km² density, valid raster-cell coverage, and first reason for inclusion/exclusion.
- `citywide_population_eligible_polygons.geojson`: polygons strictly above both population (30 people by default) and density (0 people/km² by default) thresholds.
- `citywide_population_risk_clusters.csv/.geojson`: dissolved clusters for all five gap scenarios, with estimated population, dissolved area in km², and people/km² density.
- `citywide_population_cluster_membership.csv`: polygon-to-cluster link for each scenario.
- `citywide_cluster_map_{gap}m.html`: interactive map of every selected-gap dissolved cluster, with hover details including part count, population, dissolved area, density, and span (5 m by default); choose OpenTopoMap streets and terrain or Esri satellite imagery in the layer control.
- `citywide_population_cluster_draw_frame.csv/.geojson`: all clusters at the chosen draw gap, with geographic group, random order, and selection status.
- `citywide_population_selected_clusters.csv/.geojson`: up to five primary dissolved cluster polygons and no reserves by default.
- `citywide_population_selected_boundary_polygons.geojson`: individual high-risk polygon parts belonging to selected clusters, with original source IDs.
- `citywide_cluster_size_diagnostics.png`: cluster counts by gap, member polygons, and population by cluster.
- `citywide_cluster_map_before_draw.png`: raster-backed map of the selected-gap clusters, with density labels on up to 20 populous clusters, available before the draw.
- `citywide_cluster_gap_comparison.png`: maps comparing cluster geometry and median cluster density at all five gaps.
- `citywide_population_cluster_manifest.json`: exact source filenames, screening settings, dissolved-union population totals before filtering and after selection, counts, seed, and geographic split.

The source is the specified high-risk GeoJSON and the 2026 WorldPop raster only. The notebook does not use the four-alcaldía cadastre or vulnerable-lot data. Polygon density divides estimated people by the area of the high-risk polygon, not by residential or built-up area. The printed beginning and end totals use dissolved unions to avoid double counting overlapping boundaries; the end total includes primary and reserve clusters. Cluster population is also recomputed on dissolved geometry. A polygon can belong to one cluster **per gap scenario**, so comparisons across scenarios are sensitivity analyses, not additive counts. Inspect each selected site's physical boundary and eligible parcels before treating it as a dissertation sampling location.
